In [1]:
# Cell 1
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install "torch==2.5.1+cu121" --index-url https://download.pytorch.org/whl/cu121
!{sys.executable} -m pip install "transformers>=4.41.0" "accelerate>=0.30.0" "tqdm"

import json
from pathlib import Path

import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, logging

Looking in indexes: https://download.pytorch.org/whl/cu121


Looking in indexes: https://download.pytorch.org/whl/cu121


/workspace/miniconda3/envs/punjabi-llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Cell 2
logging.set_verbosity_error()

model_name = "mistralai/Mistral-7B-Instruct-v0.3"

if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    if major >= 8:
        torch_dtype = torch.bfloat16
    else:
        torch_dtype = torch.float16
else:
    torch_dtype = torch.float16

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch_dtype,
 )
model.eval()

Loading weights: 100%|██████████| 291/291 [00:02<00:00, 127.66it/s]


Loading weights: 100%|██████████| 291/291 [00:02<00:00, 127.66it/s]


MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): MistralRMSNorm((4096,)

In [8]:
# Cell 3
data_path = Path("/workspace/FINE_TUNING_REPO/mistral_ready.jsonl")
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at {data_path}")

data = []
with data_path.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        data.append(json.loads(line))

len(data)

30

In [ ]:
# Cell 4
import re

def clean_output(text):
    text = text.replace("\ufffd", "")
    text = text.encode("utf-8", "ignore").decode("utf-8")
    lines = text.split("\n")
    cleaned = []

    for line in lines:
        if re.search(r"[A-Za-z]", line):
            continue
        if re.search(r"[\u0A00-\u0A7F]", line):
            cleaned.append(line.strip())

    return " ".join(cleaned).strip()

In [ ]:
# Cell 5
results = []
truncated = 0
max_new_tokens = 512

for sample in tqdm(data, desc="Generating"):
    prompt = sample["prompt"]
    reference = sample["reference"]

    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_ids = inputs["input_ids"][0]
    output_ids = outputs[0]
    generated_ids = output_ids[len(input_ids):]
    if len(generated_ids) == max_new_tokens:
        truncated += 1

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    response = clean_output(response)

    results.append({
        "prompt": prompt,
        "output": response,
        "reference": reference,
    })

preview = [(r["output"], r["reference"]) for r in results[:2]]
len(results), truncated, preview

Generating: 100%|██████████| 30/30 [01:58<00:00,  3.94s/it]


(30,
 24,
 [('ਡਾਇਬਟੀਜ਼ ਇੱਕ ਪੁਰਾਣੀ ਡਾਕਟਰੀ ਸਥਿਤੀ ਹੈ, ਜਿਸ ਵਿੱਚ ਸਰੀਰ ਬਲੱਡ ਸ਼ੂਗਰ ਦੇ ਪੱਧਰਾਂ ਨੂੰ ਸਹੀ ਢੰਗ ਨਾਲ ਨਿਯੰਤ੍ਰਿਤ ਕਰਨ ਵਿੱਚ',
   'ਡਾਇਬੀਟੀਜ਼ ਇੱਕ ਪੁਰਾਣੀ ਬਿਮਾਰੀ ਹੈ ਜਿੱਥੇ ਸਰੀਰ ਬਲੱਡ ਸ਼ੂਗਰ ਨੂੰ ਨਿਯੰਤ੍ਰਿਤ ਨਹੀਂ ਕਰ ਸਕਦਾ, ਜਿਸ ਨਾਲ ਦਿਲ ਦੀ ਬਿਮਾਰੀ, ਗੁਰਦੇ ਫੇਲ੍ਹ ਹੋਣਾ ਅਤੇ ਨਸਾਂ ਨੂੰ ਨੁਕਸਾਨ ਵਰਗੀਆਂ ਪੇਚੀਦਗੀਆਂ ਪੈਦਾ ਹੁੰਦੀਆਂ ਹਨ।'),
  ('ਪ੍ਰਕਾਸ਼ ਸੰਸ਼ਲੇਸ਼ਣ ਦੀ ਪ੍ਰਕਿਰਿਆ ਧਰਤੀ ਉੱਤੇ ਸਭ ਤੋਂ ਬੁਨਿਆਦੀ ਜੈਵਿਕ ਪ੍ਰਕਿਰਿਆਵਾਂ ਵਿੱਚੋਂ ਇੱਕ ਹੈ। ਇਹ ਉਹ ਵਿਧੀ ਹੈ ਜ',
   "ਪ੍ਰਕਾਸ਼ ਸੰਸ਼ਲੇਸ਼ਣ ਉਹ ਪ੍ਰਕਿਰਿਆ ਹੈ ਜਿਸ ਦੁਆਰਾ ਪੌਦੇ ਗਲੂਕੋਜ਼ ਅਤੇ ਆਕਸੀਜਨ ਪੈਦਾ ਕਰਨ ਲਈ ਸੂਰਜ ਦੀ ਰੌਸ਼ਨੀ, ਕਾਰਬਨ ਡਾਈਆਕਸਾਈਡ ਅਤੇ ਪਾਣੀ ਦੀ ਵਰਤੋਂ ਕਰਦੇ ਹਨ, ਜਿਸ ਨਾਲ ਧਰਤੀ 'ਤੇ ਲਗਭਗ ਸਾਰੇ ਜੀਵਨ ਨੂੰ ਕਾਇਮ ਰੱਖਿਆ ਜਾਂਦਾ ਹੈ।")])

In [21]:
# Cell 6
import json

out_path = Path("/workspace/FINE_TUNING_REPO/mistral_fresh_output.jsonl")
with out_path.open("w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

out_path

PosixPath('/workspace/FINE_TUNING_REPO/mistral_fresh_output.jsonl')